In [1]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
import re
from nltk.corpus import gutenberg, stopwords

In [2]:
# Utility function to clean text.

def text_cleaner(text):
    # Visual inspection shows spaCy does not recognize the double dash '--'.
    # Better get rid of it now!
    text = re.sub(r'--',' ',text)
    # Get rid of headings in square brackets.
    text = re.sub(r'<.*?>', '', text)
    # Get rid of chapter titles.
    text = re.sub(r'Chapter \d+','',text)
    # Get rid of extra whitespace.
    text = ' '.join(text.split())
    return text[0:900000]

# Import all the Austen in the Project Gutenberg corpus.
austen = ""
for novel in ['persuasion','emma','sense']:
    work = gutenberg.raw('austen-' + novel + '.txt')
    austen = austen + work

# Clean the data.
austen_clean = text_cleaner(austen)

In [3]:
%%time
# Parse the data. This can take some time.
nlp = spacy.load("en_core_web_sm")

austen_doc = nlp(austen_clean)

CPU times: user 12.9 s, sys: 1.43 s, total: 14.3 s
Wall time: 14.7 s


In [4]:
# Organize the parsed doc into sentences, while filtering out punctuation
# and stop words, and converting words to lower case lemmas.

sentences = []

for sentence in austen_doc.sents:
    sentence = [
        token.lemma_.lower()
        for token in sentence
        if not token.is_stop
        and not token.is_punct
    ]
    sentences.append(sentence)

print(sentences[20])
print('We have {} sentences and {} tokens.'.format(len(sentences), len(austen_clean)))

['elizabeth', 'succeed', 'sixteen', 'possible', 'mother', 'right', 'consequence', 'handsome', 'like', 'influence', 'great', 'go', 'happily']
We have 7570 sentences and 900000 tokens.


In [6]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 17.6 MB/s eta 0:00:0000:0100:01


In [7]:
import gensim
from gensim.models import word2vec

model = word2vec.Word2Vec(
    sentences,
    workers=4,     # Number of threads to run in parallel (if your computer does parallel processing).
    min_count=10,  # Minimum word count threshold.
    window=6,      # Number of words around target word to consider.
    sg=0,          # Use CBOW because our corpus is small.
    sample=1e-3 ,  # Penalize frequent words.
    vector_size=300,      # Word vector length.
    hs=1           # Use hierarchical softmax.
)

print('done!')

done!


In [8]:
# List of words in model.

vocab = model.wv.key_to_index

print(model.wv.most_similar(positive=['lady', 'man'], negative=['woman']))


# Similarity is calculated using the cosine, so again 1 is total
# similarity and 0 is no similarity.
print(model.wv.similarity('mr', 'mrs'))

# One of these things is not like the other...
print(model.wv.doesnt_match("breakfast marriage dinner lunch".split()))

[('goddard', 0.8269097805023193), ('louisa', 0.8099820017814636), ('colonel', 0.7854925394058228), ('excessively', 0.7815282344818115), ('benwick', 0.7790107727050781), ('miss', 0.7240598201751709), ('god', 0.7211165428161621), ('throat', 0.7159165143966675), ('weston', 0.7022116184234619), ('wentworth', 0.7010452747344971)]
0.73991257
marriage
